# CATMOSPHERE-STORM — six-basin genesis data preparation and selected models

**Scope:** corrected historical *first retained TS fix* timestamps and basin identity, 1° genesis exposure derived from the project's OSM 0.01° mask, complete six-basin `model_df`, and the **unchanged selected model formulas from the attached `basin_specific_genesis.ipynb`**. This is preparation/calibration only; it **does not** modify `EMPIRICAL`, `GPI`, `GPI-MIX`, `Basins_WMO`, or `Startingpoint`.

The workflow depends on a small new helper module, `CODE/genesis_dataset.py`, supplied with this notebook. Place both files in the CATMOSPHERE-STORM repository before execution. Historical `.npy` arrays, OSM packed mask, W500 GRIB, SST archives, and RONI CSV are not distributed in the source ZIP: configure their paths below.

Key decisions retained from the original experiments:
- Six separate GLM formulas, basin-specific vorticity clipping, basin-specific geographic soft weights.
- NA uses annually aggregated relative SST, WP uses 3-month EIO SST anomaly, EP uses `rsst_bis`, SP uses OND RONI. NI uses monsoon phase and hard regions; SI uses soft regions.
- Vorticity is **cyclonic absolute vorticity** in each hemisphere. The raw `VORT850` is relative vorticity.
- Train = years before 2016, test = years 2016 onward. Scaling and EIO SST climatology use training years only. Models are not re-selected here.

**Event definition:** the first retained 3-hourly track fix after the preprocessing TS criterion. This is the time corresponding to the stored first latitude/longitude, not necessarily the exact original first threshold-crossing timestamp. The source basin is read from `BASINLIST_INTERP.npy`, never inferred from intersecting basin bounding boxes.

## 1. Setup and data paths

This notebook may be run from the repository root or from `NOTEBOOKS/`. You can override every path with an environment variable. `PREPROC_DIR` should contain the `*LIST_INTERP.npy` dictionaries from `MASTER_preprocessing.py`.

In [ ]:
from __future__ import annotations
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

cwd = Path.cwd().resolve()
root_override = os.environ.get("CATMOSPHERE_REPO_ROOT")
if root_override:
    REPO_ROOT = Path(root_override).expanduser().resolve()
else:
    REPO_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "CODE" / "genesis_matrix.py").is_file()), None)
if REPO_ROOT is None or not (REPO_ROOT / "CODE" / "genesis_dataset.py").is_file():
    raise FileNotFoundError("Place CODE/genesis_dataset.py in the repo; set CATMOSPHERE_REPO_ROOT if needed")
sys.path.insert(0, str(REPO_ROOT))
from CODE import genesis_dataset as gd

PREPROC_DIR = Path(os.environ.get("STORM_PREPROC_DIR", str(REPO_ROOT)))
ENV_DIR = Path(os.environ.get("STORM_ENV_DIR", "/home/mbaldacchino/data/SIENA-IH-STORM_1980_2025_VF/SIENA-IH-STORM/env_yearly"))
W500_FILE = Path(os.environ.get("STORM_W500_FILE", "/home/mbaldacchino/data/vertical_velocity.grib"))
SST_HISTORY = Path(os.environ.get("STORM_SST_HISTORY", "/home/mbaldacchino/data/sst_1940_2026.nc"))
RONI_CSV = Path(os.environ.get("STORM_RONI_CSV", "/home/mbaldacchino/data/roni.csv"))
START_YEAR = int(os.environ.get("STORM_GENESIS_FIRST_YEAR", "1980"))
END_YEAR = int(os.environ.get("STORM_GENESIS_LAST_YEAR", "2025"))
FIRST_TEST_YEAR = 2016
CFG = {b: gd.GenesisConfig(basin=b, start_year=START_YEAR, end_year=END_YEAR,
                          first_test_year=FIRST_TEST_YEAR) for b in gd.BASINS}
print("Repository:", REPO_ROOT)
print("Preprocessing:", PREPROC_DIR)
print("ERA5 monthly fields:", ENV_DIR)
print("Basin months:", {b: cfg.active_months for b, cfg in CFG.items()})

## 2. Historical first-TS records (one basin per storm)

`YEARLIST_INTERP` is intentionally **not** used for the calendar year because it contains `data.season`, which can disagree with the actual date in the Southern Hemisphere. The first value in `TIMELIST_INTERP` is in **days since 1858-11-17** (as used by `preprocessing.find_month`). `BASINLIST_INTERP` stores the basin selected by the STORM preprocessing at the TS threshold; NA/EP geographic overlap is not used for reclassification.

In [ ]:
genesis_all = gd.load_first_ts_genesis(PREPROC_DIR)
assert genesis_all.storm_id.is_unique
print("Retained first-TS storms:", len(genesis_all))
print(genesis_all.groupby("basin").size().rename("retained_storms").to_string())
display(genesis_all.head())

## 3. OSM ocean exposure at 1°

Uses the project's north-up, bit-packed **0.01° OSM binary mask** (`geography/osm_land_0p01_packed.npy`). For each 1° cell, all 10,000 subpixels determine ocean fraction. The legacy coarse `<50% land` criterion is retained based on the same 10×10, 0.1° sample points as `genesis_matrix._genesis_ocean_mask_1deg()`; the NA/EP divider is additionally evaluated per 0.01° ocean subpixel when calculating eligible area.

This is **not** the 100 km land-fraction raster used for decay, and is **not** derived from `Land_ocean_masks/*.txt` or Natural Earth. 1° rows are north-to-south, and longitudes are 0–360. The mask source checksum is attached as dataset metadata.

In [ ]:
coarse_exposure = {
    basin: gd.build_osm_exposure(basin)
    for basin in gd.BASINS
}
# Audit exact coarse-mask agreement with existing GPI/EMPIRICAL compatibility rule.
from CODE import genesis_matrix as gm
for basin, ex in coarse_exposure.items():
    expected = gm._genesis_ocean_mask_1deg(basin, ex.ocean_valid.shape)
    if not np.array_equal(ex.ocean_valid.values, expected):
        n_diff = int(np.count_nonzero(ex.ocean_valid.values != expected))
        raise AssertionError(f"{basin}: {n_diff} coarse admissibility cells disagree with existing OSM STORM mask")
    assert ex.eligible_area_km2.min().item() >= 0
    print(f"{basin}: {ex.sizes['latitude']}×{ex.sizes['longitude']}; "
          f"{int(ex.ocean_valid.sum())} valid cells; "
          f"{ex.eligible_area_km2.sum().item():,.0f} eligible km²; "
          f"OSM SHA {ex.attrs['osm_sha256'][:12]}")

In [ ]:
# Optional quick map for each basin; north-up arrays can be plotted directly.
fig, axs = plt.subplots(2, 3, figsize=(16, 7), constrained_layout=True)
for ax, (basin, ds) in zip(axs.flat, coarse_exposure.items()):
    ax.imshow(ds.eligible_fraction.values, vmin=0, vmax=1,
              extent=[float(ds.longitude.min())-.5, float(ds.longitude.max())+.5,
                      float(ds.latitude.min())-.5, float(ds.latitude.max())+.5],
              origin="upper", aspect="auto")
    ax.set_title(basin)
    ax.set_xlabel("Longitude (0–360°)")
    ax.set_ylabel("Latitude")
plt.show()

## 4. Single-basin attribution and eligibility audit

An event is eligible only if its recorded basin, actual calendar year/month, 1° cell, OSM ocean point and NA/EP side all match the training domain. Every excluded record remains in `event_audit` with a specific reason. Every eligible storm occurs in exactly one `genesis[basin]`. Boundary and coastline checks are made at the actual observed coordinates, rather than on the other basin's bounding box.

In [ ]:
event_audit, genesis = gd.audit_genesis(genesis_all, coarse_exposure, CFG)
assert event_audit.storm_id.is_unique
assert sum(len(g) for g in genesis.values()) == event_audit.exclusion_reason.eq("").sum()
print(pd.crosstab(event_audit.basin, event_audit.exclusion_reason.replace("", "eligible")))
print("Total eligible observed first-TS events:", sum(map(len, genesis.values())))
display(genesis["NA"].head())

## 5. ERA5 monthly predictor tables

The helper loads the **individual** `VMAX_PI`, `RH600`, `VWS`, `VORT850` NPY files, directly sampling the exact STORM 1° cell centers from their 0.25° grids. It avoids stacking the whole globe for all years. W500 is extracted from the original monthly GRIB archive for basins whose selected models use it (all except EP). A missing historical field, ambiguous W500 timestamps, or incompatible grid raises an error rather than silently imputing climate data.

Vorticity: `eta850 = max(0, hemisphere_sign × (VORT850 + f))`. Its further clipping is performed **per basin** during feature preparation.

In [ ]:
env_1deg = {}
for basin, cfg in CFG.items():
    print("Loading", basin, "...", flush=True)
    env_1deg[basin] = gd.load_basin_environment(
        ENV_DIR, cfg, include_w500=(basin != "EP"), w500_file=W500_FILE,
    )
    print("  ", dict(env_1deg[basin].sizes), list(env_1deg[basin].data_vars))

## 6. Basin-specific climate indices (the selected models only)

NA `rsst`: JJASON mean of MDR SST minus tropical SST, region 10–25°N / 280–340°E.

EP `rsst_bis`: JJASON mean of MDR SST minus the mean of the warmest 30% of tropical SST, region 5–20°N / 220–270°E.

WP `eio_dynamic`: trailing three-month average of monthly EIO SST anomalies, region 10°S–22.5°N / 75–100°E. **Correction:** estimate each month's climatology exclusively from training years, not test years.

SP `roni_ond`: OND RONI indexed by the Southern Hemisphere season's starting calendar year. **Preserved** despite the separate research discussion about eventually avoiding direct ENSO terms. That decision can be revisited when integrating/benchmarking the model, not silently during data preparation.

In [ ]:
climate_extra = {b: None for b in gd.BASINS}
for basin in ("NA", "EP"):
    yearly_sst = gd.yearly_relative_sst(ENV_DIR, CFG[basin])
    climate_extra[basin] = gd.na_ep_climate_monthly(yearly_sst, CFG[basin])
climate_extra["WP"] = gd.wp_eio_dynamic(SST_HISTORY, CFG["WP"])
climate_extra["SP"] = gd.sp_roni_monthly(RONI_CSV, CFG["SP"])
for basin, extra in climate_extra.items():
    print(basin, "climate columns", [] if extra is None else list(extra.columns))

## 7. Construct each basin's complete `model_df` and reconcile all counts

The response is the count of eligible first-TS observations per `(calendar month, 1° cell)`; the offset is

\[ \log E_{iym}, \qquad E_{iym}=\frac{A_i^{\rm eligible}}{100\,000\,\rm km^2}\frac{D_m}{365.2425/12}. \]

Counts are joined on integer `(ilat, ilon)` indices and the exact month timestamp; no floating-point spatial join. A month/cell cannot have duplicate model rows. An eligible event with missing environmental covariates **raises**; it is not dropped. Missing-covariate ocean cells *without* events are counted in the diagnostic but excluded from fitting.

In [ ]:
model_df = {}
for basin, cfg in CFG.items():
    df = gd.build_model_table(env_1deg[basin], coarse_exposure[basin],
                              genesis[basin], cfg, extra=climate_extra[basin])
    model_df[basin] = df
    assert df.genesis_count.sum() == len(genesis[basin])
    assert not df.duplicated(["time_month", "ilat", "ilon"]).any()
    print(f"{basin}: {len(df):,} modeled cell-months, "
          f"{df.genesis_count.sum():,} eligible genesis events; "
          f"{df.attrs['n_rows_missing_covariates']:,} omitted covariate-missing ocean cell-months")

display(model_df["NA"].head())

## 8. Prepare exactly the selected basin-specific predictors

All transformations are fit using **training rows only**. The selected model's regional weights are generated from the fixed OSM eligible-cell geometry (Gaussian sigma 3° as in the basin notebooks), not from observed genesis counts. Hard `region` categories are used only for NI. All custom climate fields remain available for a later operational genesis predictor.

In [ ]:
train, test, scalers, soft_regions = {}, {}, {}, {}
for basin, cfg in CFG.items():
    df = model_df[basin]
    if basin in ("NA", "WP", "EP", "SI", "SP"):
        df, soft_regions[basin] = gd.smooth_region_weights(df, coarse_exposure[basin], cfg, sigma=3.0)
    else:
        soft_regions[basin] = []
    model_df[basin] = df
    train[basin], test[basin], scalers[basin] = gd.standardize_train_test(df, cfg)
    assert train[basin].year.max() < FIRST_TEST_YEAR <= test[basin].year.min()
    assert train[basin].genesis_count.sum() + test[basin].genesis_count.sum() == len(genesis[basin])
    print(basin, "train rows", len(train[basin]), "test rows", len(test[basin]),
          "training storms", train[basin].genesis_count.sum(), "holdout storms", test[basin].genesis_count.sum())

## 9. Keep the six final selected model specifications

These formulas are **transcribed from the final cells of the attached `basin_specific_genesis.ipynb`**, not re-optimized or replaced by one universal formula. `WP` is the repository's western North Pacific (`WNP`). An EP clipped-vorticity threshold of `3.5e-5` was recovered from its basin notebook because the consolidated EP section lacked the clipping assignment.

The SP formula deliberately still uses RONI (the notebook's selected specification). All six regressions use the same OSM-consistent exposure offset.

In [ ]:
def formula_for_basin(basin: str) -> str:
    terms = [f"w_{r}" for r in soft_regions[basin] if r != gd.REGION_REFERENCE[basin]]
    if basin == "NA":
        return "genesis_count ~ C(month) + z_pi + z_rh600 + z_vws + z_eta_clip + z_w500" + (
            " + " + " + ".join(terms) if terms else "") + " + rsst"
    if basin == "WP":
        return "genesis_count ~ z_pi + z_rh600 + z_vws + z_eta_clip + z_w500" + (
            " + " + " + ".join(terms) if terms else "") + " + z_eio_dynamic"
    if basin == "EP":
        return "genesis_count ~ z_pi + z_rh600 + z_vws + z_eta_clip + rsst_bis" + (
            " + " + " + ".join(terms) if terms else "")
    if basin == "NI":
        return "genesis_count ~ C(ni_season) + C(region) + z_eta_clip + z_vws + z_w500"
    if basin == "SI":
        return "genesis_count ~ z_pi + z_rh600 + z_vws + z_eta_clip + z_w500" + (
            " + " + " + ".join(terms) if terms else "")
    if basin == "SP":
        return ("genesis_count ~ z_pi:w_central_south_pacific + z_pi:w_eastern_south_pacific"
                " + z_vws + z_eta_clip + z_w500 + w_eastern_south_pacific"
                " + roni_ond_standardized:w_eastern_south_pacific")
    raise ValueError(basin)

FORMULAS = {basin: formula_for_basin(basin) for basin in gd.BASINS}
for basin, form in FORMULAS.items():
    print(basin, "->", form)

In [ ]:
# Fit only the six already-selected models. No search or tuning in this notebook.
models = {}
for basin in gd.BASINS:
    models[basin] = smf.glm(
        formula=FORMULAS[basin], data=train[basin],
        family=sm.families.Poisson(), offset=train[basin].log_exposure,
    ).fit()
    print(f"{basin}: {models[basin].nobs:,} rows; {len(models[basin].params)} coefficients; AIC={models[basin].aic:.1f}")

## 10. Final count audit and diagnostic artifacts

Verify that each OSM-eligible observed event remains counted once, in exactly one basin and exactly one calendar month. The optional parquet/NetCDF export provides model-ready data and the exact OSM grids for subsequent calibration/integration; exporting does not modify existing genesis modes.

In [ ]:
audit = pd.DataFrame([{
    "basin": b,
    "observed_eligible": len(genesis[b]),
    "observed_on_model_grid": int(model_df[b].genesis_count.sum()),
    "train": int(train[b].genesis_count.sum()),
    "test": int(test[b].genesis_count.sum()),
    "grid_rows": len(model_df[b]),
    "missing_covariate_ocean_rows": model_df[b].attrs.get("n_rows_missing_covariates", 0),
    "osm_sha256": coarse_exposure[b].attrs["osm_sha256"],
} for b in gd.BASINS])
assert (audit.observed_eligible == audit.observed_on_model_grid).all()
assert (audit.observed_eligible == audit.train + audit.test).all()
assert genesis_all.storm_id.is_unique and event_audit.storm_id.is_unique
assert pd.concat(genesis.values()).storm_id.is_unique
assert event_audit.exclusion_reason.eq("").sum() == audit.observed_eligible.sum()
display(audit)

SAVE_PREPARED = False   # Set True to export; outputs under REPO_ROOT/genesis_prepared
if SAVE_PREPARED:
    target = REPO_ROOT / "genesis_prepared"
    target.mkdir(exist_ok=True)
    event_audit.to_parquet(target / "event_audit.parquet", index=False)
    audit.to_csv(target / "model_reconciliation.csv", index=False)
    for b in gd.BASINS:
        model_df[b].to_parquet(target / f"model_df_{b}.parquet", index=False)
        coarse_exposure[b].to_netcdf(target / f"osm_genesis_exposure_{b}_1deg.nc")
    print("Saved:", target)

### Before the simulation integration

- This notebook reproduces the selected **formula structures**, not the historical coefficients. Retraining on corrected timestamps, source-basin attribution, exact OSM exposure and train-only climate normalization may change coefficient values and likelihoods.
- The OSM `.npy` and ERA5/IBTrACS inputs are **not included** in the repository ZIP used for this repair; run and inspect the final audit on your machine. The unit tests bundled with the repair use synthetic input data.
- For any future IPP sampler, use `coarse_exposure[basin].eligible_area_km2` as the monthly spatial offset and sample the final point within the **same OSM-ocean, NA/EP-subpixel eligible domain**. Applying a different coastline rule later would change the frequency measure.
- The historical active-month definitions and the direct RONI predictor are kept exactly as in the selected notebook, even though they may eventually be reconsidered for V1 SEAS5 availability. No current GPI/EMPIRICAL mode is touched.